# ZemaVoice Stage 3 — Experimental Amharic TTS training
**Runs on Google Colab Free.** Upload your `zemavoice_amharic_sample.zip` from iPhone Files. This is a small character-to-spectrogram proof of concept, **not a production TTS system**. FLEURS data is CC BY 4.0; retain attribution. Do not clone or impersonate identifiable speakers. Model outputs may be unintelligible. GPU recommended.

In [ ]:
!pip -q install 'librosa>=0.10,<0.12' 'soundfile>=0.12,<1' 'numpy<3'


In [ ]:
import torch, numpy as np, librosa, soundfile as sf, json, zipfile, os, random
from pathlib import Path
from google.colab import files
print("GPU:",torch.cuda.is_available(),torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
uploaded=files.upload()
assert uploaded, "Please select zemavoice_amharic_sample.zip from Files"
archive=next((name for name in uploaded if name.lower().endswith(".zip")),None)
assert archive, "Please upload the dataset ZIP file"
root=Path("/content/zemavoice_stage3"); root.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for item in z.infolist():
        dest=(root/item.filename).resolve()
        assert dest.is_relative_to(root.resolve()),"Unsafe archive path"
    z.extractall(root)
rows=[json.loads(x) for x in (root/"manifest.jsonl").read_text(encoding="utf-8").splitlines() if x.strip()]
print("Dataset clips:",len(rows))
assert len(rows)>=10, "Too few audio clips"


In [ ]:
import torch.nn as nn, torch.nn.functional as F
torch.manual_seed(7); random.seed(7)
device="cuda" if torch.cuda.is_available() else "cpu"
# Bound length and clip count for a free Colab proof of concept.
N_CLIPS=min(80,len(rows)); SR=16000; HOP=256; N_FFT=1024; N_MELS=64; MAX_TEXT=256; MAX_FRAMES=512
chosen=rows[:N_CLIPS]
chars=sorted(set("".join(r["transcript"][:MAX_TEXT] for r in chosen)))
stoi={c:i+1 for i,c in enumerate(chars)}
print("Vocabulary:",len(stoi))
data=[]
for r in chosen:
    wav_path=root/"audio"/Path(r["audio_path"]).name
    if not wav_path.exists(): continue
    y,sr=sf.read(str(wav_path),dtype="float32")
    if y.ndim>1:y=y.mean(axis=1)
    if sr!=SR:y=librosa.resample(y,orig_sr=sr,target_sr=SR)
    y=y[:MAX_FRAMES*HOP]
    if len(y)<SR:continue
    mel=librosa.feature.melspectrogram(y=y,sr=SR,n_fft=N_FFT,hop_length=HOP,n_mels=N_MELS)
    target=np.log(np.maximum(mel,1e-5)).T[:MAX_FRAMES]
    ids=[stoi[c] for c in r["transcript"][:MAX_TEXT] if c in stoi]
    if not ids:continue
    data.append((torch.tensor(ids,dtype=torch.long),torch.tensor(target,dtype=torch.float32)))
print("Prepared examples:",len(data));assert len(data)>=10


In [ ]:
class TinyAmharicTTS(nn.Module):
    def __init__(self,vocab):
        super().__init__()
        self.embed=nn.Embedding(vocab+1,96,padding_idx=0)
        self.encoder=nn.GRU(96,128,batch_first=True,bidirectional=True)
        self.decoder=nn.Sequential(nn.Linear(256,256),nn.ReLU(),nn.Linear(256,N_MELS))
    def forward(self,ids,frames):
        enc,_=self.encoder(self.embed(ids.unsqueeze(0)))
        # Simple linear text-to-frame stretching, NOT learned phoneme durations.
        x=F.interpolate(enc.transpose(1,2),size=frames,mode="linear",align_corners=False).transpose(1,2)
        return self.decoder(x).squeeze(0)
model=TinyAmharicTTS(len(stoi)).to(device)
optim=torch.optim.Adam(model.parameters(),lr=0.001)
random.shuffle(data)
train=data[:-8]; valid=data[-8:]
print("Training",len(train),"examples, validation",len(valid))
for epoch in range(4):
    model.train(); losses=[]
    for ids,target in train:
        ids,target=ids.to(device),target.to(device)
        optim.zero_grad()
        pred=model(ids,len(target))
        loss=F.mse_loss(pred,target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optim.step(); losses.append(loss.item())
    model.eval()
    with torch.no_grad():
        vl=[F.mse_loss(model(ids.to(device),len(target)),target.to(device)).item() for ids,target in valid]
    print(f"Epoch {epoch+1}/4 — train loss {np.mean(losses):.4f} — validation loss {np.mean(vl):.4f}")
print("Training experiment finished")


In [ ]:
from IPython.display import Audio,display
model.eval()
# Test an unseen phrase. The model has no duration predictor; length is heuristic.
phrase="ሰላም እንዴት ናችሁ"
ids=torch.tensor([stoi.get(c,0) for c in phrase],dtype=torch.long,device=device)
frames=min(MAX_FRAMES,max(50,len(phrase)*12))
with torch.no_grad(): logmel=model(ids,frames).cpu().numpy().T
mel=np.exp(np.clip(logmel,-12,10))
wave=librosa.feature.inverse.mel_to_audio(mel,sr=SR,n_fft=N_FFT,hop_length=HOP,n_iter=24)
sf.write("/content/zemavoice_first_experiment.wav",wave,SR)
print("Generated experimental WAV (quality is not guaranteed):")
display(Audio(wave,rate=SR))


In [ ]:
weights="/content/zemavoice_experimental_weights.pt"
torch.save({"state_dict":model.cpu().state_dict(),"vocab":stoi,"sample_rate":SR,"architecture":"TinyAmharicTTS proof-of-concept; fixed text-frame interpolation","dataset":"Google FLEURS am_et CC BY 4.0"},weights)
print("Saved model:",weights)
files.download(weights)
files.download("/content/zemavoice_first_experiment.wav")


## Important
The model trains a rudimentary spectrogram predictor and uses Griffin–Lim reconstruction. **It is not expected to produce intelligible, natural Amharic speech.** This is a GPU/training-loop verification. Production development requires a licensed speech model architecture, alignment/duration modeling, vocoder, much more high-quality consent-cleared data, and speaker evaluations. Free Colab sessions may disconnect; download the weights promptly.